<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/6_Bidirectional_Long_Short_Term_Memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import PyTorch for tensor computations.
# Only tensor operations are used; nn.LSTM and nn.RNN are not used.
import torch

# Import NumPy for numerical computations.
import numpy as np

# Set random seeds for reproducible BiLSTM results.
torch.manual_seed(42)
np.random.seed(42)

In [2]:
# Sigmoid activation is used by the Input,
# Forget, and Output gates to control information flow.

def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

# Tanh activation generates the Cell State Candidate
# and compresses hidden state values between -1 and 1.

def tanh(x):
    return torch.tanh(x)

In [3]:
# Initialize weight matrices and bias vectors
# for all four LSTM gates.

def initialize_parameters(input_size, hidden_size):

    params = {}

    # Input Gate parameters.
    params["Wi"] = torch.randn(input_size + hidden_size, hidden_size)
    params["bi"] = torch.zeros(hidden_size)

    # Forget Gate parameters.
    params["Wf"] = torch.randn(input_size + hidden_size, hidden_size)
    params["bf"] = torch.zeros(hidden_size)

    # Output Gate parameters.
    params["Wo"] = torch.randn(input_size + hidden_size, hidden_size)
    params["bo"] = torch.zeros(hidden_size)

    # Cell Candidate parameters.
    params["Wg"] = torch.randn(input_size + hidden_size, hidden_size)
    params["bg"] = torch.zeros(hidden_size)

    return params

In [4]:
# Compute one LSTM time-step using the
# Input Gate, Forget Gate, Output Gate,
# and Cell State Candidate.

def lstm_cell(x, h_prev, c_prev, params):

    # Combine current input and previous hidden state.
    combined = torch.cat((x, h_prev), dim=1)

    # Input Gate controls new information entering memory.
    i = sigmoid(combined @ params["Wi"] + params["bi"])

    # Forget Gate controls information removed from memory.
    f = sigmoid(combined @ params["Wf"] + params["bf"])

    # Output Gate controls information exposed
    # from the hidden state.
    o = sigmoid(combined @ params["Wo"] + params["bo"])

    # Cell Candidate generates new memory content.
    g = tanh(combined @ params["Wg"] + params["bg"])

    # Update the Cell State.
    c = f * c_prev + i * g

    # Compute the Hidden State.
    h = o * tanh(c)

    return h, c

In [5]:
# Process the sequence from the first
# time step to the last time step.

def forward_lstm(sequence, params, hidden_size):

    h = torch.zeros(1, hidden_size)

    c = torch.zeros(1, hidden_size)

    outputs = []

    for t in range(sequence.shape[0]):

        # Compute one forward LSTM step.
        h, c = lstm_cell(
            sequence[t:t+1],
            h,
            c,
            params
        )

        outputs.append(h)

    return outputs

In [6]:
# Process the sequence from the last
# time step to the first time step.

def backward_lstm(sequence, params, hidden_size):

    h = torch.zeros(1, hidden_size)

    c = torch.zeros(1, hidden_size)

    outputs = []

    for t in reversed(range(sequence.shape[0])):

        # Compute one backward LSTM step.
        h, c = lstm_cell(
            sequence[t:t+1],
            h,
            c,
            params
        )

        outputs.insert(0, h)

    return outputs

In [7]:
# Execute both forward and backward LSTM passes.
# Concatenate hidden states from both directions
# to capture past and future sequence context.

def bidirectional_lstm(sequence, forward_params, backward_params, hidden_size):

    forward_output = forward_lstm(
        sequence,
        forward_params,
        hidden_size
    )

    backward_output = backward_lstm(
        sequence,
        backward_params,
        hidden_size
    )

    outputs = []

    for f, b in zip(forward_output, backward_output):

        outputs.append(torch.cat((f, b), dim=1))

    return outputs

In [8]:
# Generate a sequence of feature vectors.
# Each row represents one time step.

sequence = torch.randn(8,5)

In [9]:
# Create independent parameters
# for both LSTM directions.

forward_params = initialize_parameters(5,4)

backward_params = initialize_parameters(5,4)

In [10]:
# Perform bidirectional sequence processing.

outputs = bidirectional_lstm(
    sequence,
    forward_params,
    backward_params,
    hidden_size=4
)

In [11]:
# Verify the hidden representation
# generated at every sequence step.

for i, output in enumerate(outputs):

    print(f"Time Step {i+1} Output Shape :", output.shape)

Time Step 1 Output Shape : torch.Size([1, 8])
Time Step 2 Output Shape : torch.Size([1, 8])
Time Step 3 Output Shape : torch.Size([1, 8])
Time Step 4 Output Shape : torch.Size([1, 8])
Time Step 5 Output Shape : torch.Size([1, 8])
Time Step 6 Output Shape : torch.Size([1, 8])
Time Step 7 Output Shape : torch.Size([1, 8])
Time Step 8 Output Shape : torch.Size([1, 8])
